# Simulación de Álbum de Estampas

Este notebook combina dos simulaciones:
- **Lab 8-3:** Impacto del mercado de intercambio (K repetidas por faltante)
- **Lab 8-4:** Impacto de un presupuesto fijo de Q1,000

## Configuración general

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros compartidos
N = 100       # Estampas totales en el álbum
S = 7         # Estampas por sobre
R = 10_000    # Repeticiones de la simulación
np.random.seed(2026)

: 

---
## Lab 8-3 – Mercado de Intercambio

Se modela la posibilidad de canjear **K estampas repetidas** por **1 estampa faltante**.
Se comparan cuatro tasas de intercambio (K = 1, 2, 5, 10) contra el caso base (sin intercambio).

### Parámetros y simulación

In [ ]:
K_values = [1, 2, 5, 10]
M_values = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]

# K_all: None representa 'Sin Intercambio'
K_all = [None] + K_values

resultados_sim = {}

print('Iniciando simulaciones de mercado de intercambio...')

for K in K_all:
    sobres_totales = np.zeros(R)

    for i in range(R):
        album = np.zeros(N, dtype=bool)
        repetidas = 0
        sobres = 0

        while True:
            faltantes = N - album.sum()

            # Condición de victoria
            if K is None:
                if faltantes == 0:
                    break
            else:
                # Las repetidas alcanzan para canjear todas las faltantes
                if faltantes == 0 or faltantes <= repetidas // K:
                    break

            sobre = np.random.choice(N, size=S, replace=False)
            nuevas = sobre[~album[sobre]]
            repetidas += S - len(nuevas)
            album[nuevas] = True
            sobres += 1

        sobres_totales[i] = sobres

    resultados_sim[K] = sobres_totales

print('Simulación completada.')

### Parte A – Impacto estadístico del intercambio

In [ ]:
media_base = np.mean(resultados_sim[None])

print('=' * 50)
print(' PARTE A: IMPACTO DEL INTERCAMBIO HASTA COMPLETAR')
print('=' * 50)
print(f'Sin Intercambio  | Media: {media_base:.2f} sobres')
print('-' * 50)

for K in K_values:
    media = np.mean(resultados_sim[K])
    std   = np.std(resultados_sim[K], ddof=1)
    reduccion = ((media_base - media) / media_base) * 100
    print(f'K = {K:<2}           | Media: {media:.2f} sobres | Desv: {std:.2f} | Ahorro: {reduccion:.2f}%')

print('=' * 50)

#### Histograma – distribución de sobres necesarios

In [ ]:
colores   = {None: 'gray', 1: 'purple', 2: 'green', 5: 'orange', 10: 'red'}
etiquetas = {None: 'Sin Intercambio', 1: 'K=1', 2: 'K=2', 5: 'K=5', 10: 'K=10'}

plt.figure(figsize=(10, 6))

for K in reversed(K_all):
    plt.hist(resultados_sim[K], bins=range(10, 120), alpha=0.6,
             color=colores[K], label=etiquetas[K], edgecolor='black', linewidth=0.5)

plt.title('Distribución de Sobres Necesarios según Tasa de Intercambio (K)', fontsize=14)
plt.xlabel('Sobres Comprados', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.xlim(10, 110)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Parte B – Sobres necesarios para distintos niveles de certeza

In [ ]:
print('=' * 50)
print(' PARTE B: SOBRES NECESARIOS PARA ALCANZAR CERTEZA')
print('=' * 50)

for K in K_all:
    etiqueta = 'Sin Intercambio' if K is None else f'K = {K}'
    p50 = np.percentile(resultados_sim[K], 50)
    p75 = np.percentile(resultados_sim[K], 75)
    p90 = np.percentile(resultados_sim[K], 90)
    print(f'{etiqueta:<15} -> 50%: {int(p50)} sobres | 75%: {int(p75)} sobres | 90%: {int(p90)} sobres')

#### Curvas de probabilidad de completar el álbum con M sobres

In [ ]:
plt.figure(figsize=(10, 6))

for K in K_all:
    probs  = [np.mean(resultados_sim[K] <= m) for m in M_values]
    estilo = '--' if K is None else '-'
    plt.plot(M_values, probs, marker='o', linestyle=estilo, linewidth=2.5,
             color=colores[K], label=etiquetas[K])

plt.axhline(y=0.5, color='black', linestyle=':', alpha=0.5, label='50% Probabilidad')
plt.axhline(y=0.9, color='black', linestyle=':', alpha=0.5, label='90% Probabilidad')

plt.title('Probabilidad de Completar el Álbum comprando M sobres', fontsize=14)
plt.xlabel('Cantidad Exacta de Sobres Comprados (M)', fontsize=12)
plt.ylabel('Probabilidad de Éxito', fontsize=12)
plt.xticks(M_values)
plt.yticks(np.arange(0, 1.1, 0.1))
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

---
## Lab 8-4 – Restricción de Presupuesto

Se simula la compra de sobres con un **presupuesto máximo de Q1,000**.
Se calcula la probabilidad de completar el álbum y el progreso promedio en caso de fracaso.

### Parámetros y simulación

In [ ]:
P_SOBRE    = 9.50      # Precio por sobre (Q)
PRESUPUESTO = 1_000.0  # Presupuesto total (Q)

max_sobres_permitidos = int(PRESUPUESTO // P_SOBRE)

completados          = 0
historial_sobres     = np.zeros(R)
estampas_en_fracasos = []

print('Simulando con límite de presupuesto...')

for i in range(R):
    album           = np.zeros(N, dtype=bool)
    sobres_comprados = 0
    gasto_acumulado  = 0.0

    while (gasto_acumulado + P_SOBRE <= PRESUPUESTO) and (album.sum() < N):
        sobre = np.random.choice(N, size=S, replace=False)
        album[sobre] = True
        sobres_comprados += 1
        gasto_acumulado  += P_SOBRE

    historial_sobres[i] = sobres_comprados

    if album.sum() == N:
        completados += 1
    else:
        estampas_en_fracasos.append(int(album.sum()))

print('Simulación completada.')

### Resultados

In [ ]:
probabilidad_exito   = completados / R
probabilidad_fracaso = 1 - probabilidad_exito
media_sobres          = np.mean(historial_sobres)
media_estampas_fracaso = np.mean(estampas_en_fracasos) if estampas_en_fracasos else 0

print('=' * 50)
print('     RESULTADOS FINANCIEROS (Presupuesto Q1,000)')
print('=' * 50)
print(f'Máximo de sobres posibles con Q1,000: {max_sobres_permitidos} sobres')
print(f'Probabilidad de COMPLETAR el álbum:   {probabilidad_exito*100:.2f}%')
print(f'Probabilidad de FRACASAR:             {probabilidad_fracaso*100:.2f}%')
print('-' * 50)
print(f'Sobres comprados en promedio:         {media_sobres:.1f} sobres')
print(f'Estampas únicas al fracasar (Media):  {media_estampas_fracaso:.1f} estampas')
print('=' * 50)

#### Gráfico de barras – proporción de éxito vs fracaso

In [ ]:
etiquetas_bar = ['Completado (Éxito)', 'Incompleto (Fracaso)']
valores       = [probabilidad_exito, probabilidad_fracaso]
colores_bar   = ['#4CAF50', '#F44336']

plt.figure(figsize=(8, 6))
barras = plt.bar(etiquetas_bar, valores, color=colores_bar, edgecolor='black', alpha=0.8)

for barra, valor in zip(barras, valores):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.01,
        f'{valor*100:.2f}%',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

plt.title(
    f'Proporción de Éxito al Llenar el Álbum\n'
    f'(Presupuesto de Q1,000 = Máx. {max_sobres_permitidos} sobres)',
    fontsize=14
)
plt.ylabel('Proporción (0 a 1)', fontsize=12)
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()